# BARAM 2026 — FiLM 그룹 조건화 실험 (계층적 MLP) → v5

**질문**: 발전단지(그룹)가 3개인데, MLP의 그룹 조건화를 concat 임베딩 대신 **FiLM으로 계층적으로** 하면 나아지는가?

리서치 문서(research_conditioning_layers) 권고의 실전 적용:
- 이산 조건(그룹 3개) → **per-group (γ_g, β_g) 테이블**로 은닉층을 변조 (class-conditional 변조의 검증된 패턴)
- **FiLM-Zero**: γ=1, β=0 초기화 → 학습 시작 시 무조건화 모델과 동일 (최소한 나빠지지 않는 안전장치)

**비교 (v4 튜닝 설정 256×3 고정, 조건화만 변경)**
| 변형 | 조건화 |
|---|---|
| A_concat | 입력에 그룹 임베딩 concat (v4 현행 = 대조군) |
| B_film | 층별 FiLM-Zero (γ_g, β_g), 입력 임베딩 없음 |
| C_film_concat | FiLM + concat 병용 |
| D_heads | 공유 몸통 + 그룹별 출력 헤드 (고전적 계층 구조) |

평가: CV 2폴드(→2023, →2024), GBM 블렌드 w∈{0.5,0.6,0.7}, 공식 총점. **채택 = 두 해 모두 A를 이길 때만.**
MLP 학습은 **MPS**(가능 시) 사용.

## 0. 설정 & GBM 폴드 캐시

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores, metric

DEV = "mps" if torch.backends.mps.is_available() else "cpu"
print("torch device:", DEV)

GROUPS=(1,2,3); FR={}; TGT={}; FR_TE={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
V2=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256)   # v4 튜닝 설정

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def gbm_ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

YEAR_FOLDS=[([2022],2023),([2022,2023],2024)]
CACHE={}
for tys,vy in YEAR_FOLDS:
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        ent[g]=dict(tr2=W.with_pc(tr,iso),va2=W.with_pc(va,iso))
        ent[g]["gbm"]=np.clip(gbm_ens(ent[g]["tr2"],ent[g]["va2"],V2,tgt),0,cap)
    CACHE[vy]=ent
    print(f"cached →{vy}")

def fold_total(preds,vy):
    nm=[]; fi=[]
    for g,p in preds.items():
        tgt=TGT[g]; cap=W.CAP[g]; yt=CACHE[vy][g]["va2"][tgt].to_numpy()
        a,b=group_scores(yt,np.clip(p,0,cap),cap); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)
GBM_TOTAL={vy:fold_total({g:CACHE[vy][g]["gbm"] for g in CACHE[vy]},vy) for vy in (2023,2024)}
print("GBM 단독:",{k:round(v,4) for k,v in GBM_TOTAL.items()})

torch device: mps


cached →2023


cached →2024
GBM 단독: {2023: np.float64(0.5862), 2024: np.float64(0.6013)}


## 1. 조건화 변형 4종

In [2]:
class CondMLP(nn.Module):
    """mode: concat | film | film_concat | heads. FiLM은 zero-init(항등 시작)."""
    def __init__(s,nf,mode,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.mode=mode; s.depth=depth
        use_cat = mode in ("concat","film_concat")
        s.emb = nn.Embedding(ng,emb) if use_cat else None
        d_in = nf + (emb if use_cat else 0)
        s.lins=nn.ModuleList([nn.Linear(d_in if i==0 else h, h) for i in range(depth)])
        s.act=nn.GELU(); s.drop=nn.Dropout(drop)
        if mode in ("film","film_concat"):
            s.film=nn.ModuleList([nn.Embedding(ng,2*h) for _ in range(depth)])
            for f in s.film: nn.init.zeros_(f.weight)      # γ=1+0, β=0 → 항등
        else: s.film=None
        if mode=="heads":
            s.heads=nn.Embedding(ng,h+1)                    # 그룹별 (w,b) 헤드
            nn.init.zeros_(s.heads.weight)
            s.shared_head=nn.Linear(h,1)                    # 공유 헤드 + 그룹별 잔차(zero-init)
        else:
            s.head=nn.Linear(h,1)
    def forward(s,x,g):
        if s.emb is not None: x=torch.cat([x,s.emb(g)],-1)
        hdd=x
        for i,lin in enumerate(s.lins):
            z=lin(hdd)
            if s.film is not None:
                gb=s.film[i](g); gamma,beta=gb.chunk(2,-1)
                z=(1+gamma)*z+beta
            hdd=s.drop(s.act(z))
        if s.mode=="heads":
            wb=s.heads(g); w_,b_=wb[...,:-1],wb[...,-1]
            return s.shared_head(hdd).squeeze(-1) + (hdd*w_).sum(-1)+b_
        return s.head(hdd).squeeze(-1)

def train_mlp(pool_tr,feats,mode,seed=15):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV); gt=torch.tensor(gid,device=DEV)
    net=CondMLP(len(feats),mode,h=CFG["h"],depth=CFG["depth"],drop=CFG["drop"]).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=CFG["lr"],weight_decay=CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),CFG["bs"]):
            b=torch.tensor(perm[i:i+CFG["bs"]],device=DEV)
            opt.zero_grad(); ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def mlp_predict(net,scaler,fr,feats,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def make_pool(vy):
    parts=[]
    for g,e in CACHE[vy].items():
        p=e["tr2"][V2+["kst_dtm"]].copy(); p["cf"]=e["tr2"][TGT[g]]/W.CAP[g]; p["gid"]=g-1; parts.append(p)
    return pd.concat(parts,ignore_index=True)

## 2. 4변형 × 2폴드 × 블렌드 w 평가

In [3]:
MODES=["concat","film","film_concat","heads"]; WEIGHTS=[0.5,0.6,0.7]
res={}
for mode in MODES:
    tot={}
    for vy in (2023,2024):
        net,scaler=train_mlp(make_pool(vy),V2,mode)
        preds={g:mlp_predict(net,scaler,CACHE[vy][g]["va2"],V2,g,W.CAP[g]) for g in CACHE[vy]}
        tot[vy]={w:fold_total({g:(1-w)*CACHE[vy][g]["gbm"]+w*preds[g] for g in CACHE[vy]},vy) for w in WEIGHTS}
    res[mode]=tot
    bw=max(WEIGHTS,key=lambda w:0.5*(tot[2023][w]+tot[2024][w]))
    print(f"{mode:12s}: best w={bw}  2023={tot[2023][bw]:.4f}  2024={tot[2024][bw]:.4f}  mean={0.5*(tot[2023][bw]+tot[2024][bw]):.4f}")

# 판정: A_concat 대비 두 해 모두 우위인 변형 중 평균 최대
base=res["concat"]
bw0=max(WEIGHTS,key=lambda w:0.5*(base[2023][w]+base[2024][w]))
base23,base24=base[2023][bw0],base[2024][bw0]
adopt=None; adopt_w=None; adopt_mean=-1
for mode in ["film","film_concat","heads"]:
    for w in WEIGHTS:
        t23,t24=res[mode][2023][w],res[mode][2024][w]
        if t23>base23 and t24>base24 and 0.5*(t23+t24)>adopt_mean:
            adopt,adopt_w,adopt_mean=mode,w,0.5*(t23+t24)
print(f"\nA_concat 기준: w={bw0}, 2023={base23:.4f}, 2024={base24:.4f}")
print("판정:", f"{adopt} (w={adopt_w}, mean={adopt_mean:.4f}) 채택" if adopt else "concat 유지 (FiLM/계층 변형이 두 해 모두 이기지 못함)")
summary=dict(gbm_total={str(k):round(v,4) for k,v in GBM_TOTAL.items()},
    results={m:{str(vy):{str(w):round(res[m][vy][w],4) for w in WEIGHTS} for vy in (2023,2024)} for m in MODES},
    adopted=adopt or "concat(v4 유지)", adopted_w=adopt_w)
json.dump(summary,open("film_ablation_summary.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

concat      : best w=0.7  2023=0.6011  2024=0.6139  mean=0.6075


film        : best w=0.5  2023=0.5899  2024=0.6101  mean=0.6000


film_concat : best w=0.6  2023=0.5909  2024=0.6085  mean=0.5997


heads       : best w=0.6  2023=0.5982  2024=0.6124  mean=0.6053

A_concat 기준: w=0.7, 2023=0.6011, 2024=0.6139
판정: concat 유지 (FiLM/계층 변형이 두 해 모두 이기지 못함)
{
  "gbm_total": {
    "2023": 0.5862,
    "2024": 0.6013
  },
  "results": {
    "concat": {
      "2023": {
        "0.5": 0.5996,
        "0.6": 0.6004,
        "0.7": 0.6011
      },
      "2024": {
        "0.5": 0.6112,
        "0.6": 0.6123,
        "0.7": 0.6139
      }
    },
    "film": {
      "2023": {
        "0.5": 0.5899,
        "0.6": 0.5891,
        "0.7": 0.588
      },
      "2024": {
        "0.5": 0.6101,
        "0.6": 0.6103,
        "0.7": 0.61
      }
    },
    "film_concat": {
      "2023": {
        "0.5": 0.5907,
        "0.6": 0.5909,
        "0.7": 0.5904
      },
      "2024": {
        "0.5": 0.6076,
        "0.6": 0.6085,
        "0.7": 0.608
      }
    },
    "heads": {
      "2023": {
        "0.5": 0.5975,
        "0.6": 0.5982,
        "0.7": 0.596
      },
      "2024": {
        "0.5": 0.6111,
 

## 3. (채택 시) v5 최종 제출

FiLM/계층 변형이 이기면 아래 셀이 v5 제출을 생성. 아니면 v4 유지가 결론.

In [4]:
if adopt:
    BLEND=adopt_w; MODE=adopt
    def blend_predict(tr_frames):
        pool=[]
        for g,(tr2,_) in tr_frames.items():
            p=tr2[V2+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; pool.append(p)
        net,scaler=train_mlp(pd.concat(pool,ignore_index=True),V2,MODE)
        out={}
        for g,(tr2,va2) in tr_frames.items():
            cap=W.CAP[g]
            gp=np.clip(gbm_ens(tr2,va2,V2,TGT[g]),0,cap)
            mp=mlp_predict(net,scaler,va2,V2,g,cap)
            out[g]=np.clip((1-BLEND)*gp+BLEND*mp,0,cap)
        return out
    tr_frames={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
        m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
        iso=W.fit_powercurve(fr[m_tr],tgt,cap)
        tr_frames[g]=(W.with_pc(fr[m_tr],iso),W.with_pc(fr[m_va],iso))
    val_pred=blend_predict(tr_frames)
    OOF={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; tr2=tr_frames[g][0]
        oof=np.full(len(tr2),np.nan); years=sorted(tr2.kst_dtm.dt.year.unique())
        if len(years)>=2:
            for ty in years:
                m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
                oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
        else:
            n=len(tr2); cut=int(n*0.7)
            oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
        OOF[g]=oof
    def debias_fit(tr,tgt,oof):
        d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
        d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
        d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
        return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
    def debias_apply(va,pred,tbl,glob):
        v=va.copy()
        v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
        v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
        return pred+np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
    def nudge_fit(tr,tgt,oof,cap):
        d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
        yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy(); best=(1.0,0.0); bf=-1
        for s in np.linspace(0.90,1.15,26):
            for sh in np.linspace(-0.06,0.06,25)*cap:
                f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
                if f>bf: bf=f; best=(s,sh)
        return best
    STORE={}; choice={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; tr2,va2=tr_frames[g]; bp=val_pred[g]
        tbl,glob=debias_fit(tr2,tgt,OOF[g]); s,sh=nudge_fit(tr2,tgt,OOF[g],cap)
        p1=np.clip(debias_apply(va2,bp,tbl,glob),0,cap)
        cand={"P0_none":bp,"P1_debias":p1,"P3_nudge":np.clip(bp*s+sh,0,cap),
              "P5_deb_nudge":np.clip(p1*s+sh,0,cap)}
        STORE[g]=dict(tbl=tbl,glob=glob,nudge=(s,sh))
        rows=[]
        for name,p in cand.items():
            nm,fi=group_scores(va2[tgt].to_numpy(),p,cap); rows.append(dict(post=name,contrib=fi-nm))
        t=pd.DataFrame(rows).set_index("post"); choice[g]=t["contrib"].idxmax()
    def apply_post(g,frame,pred):
        cap=W.CAP[g]; st=STORE[g]; ch=choice[g]
        if ch=="P0_none": return pred
        if ch=="P1_debias": return np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
        if ch=="P3_nudge": return np.clip(pred*st["nudge"][0]+st["nudge"][1],0,cap)
        q=np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
        return np.clip(q*st["nudge"][0]+st["nudge"][1],0,cap)
    ans=pd.DataFrame({f"kpx_group_{g}":tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
    p1df=pd.DataFrame({f"kpx_group_{g}":apply_post(g,tr_frames[g][1],val_pred[g]) for g in GROUPS})
    t1=metric(ans,p1df)
    print(f"v5 holdout 총점(후처리 포함): {t1[0]:.4f} (1-NMAE {t1[1]:.4f}, FICR {t1[2]:.4f})  cf. v4=0.6463")
    if t1[0]>0.6463:
        full_frames={}
        for g in GROUPS:
            tgt=TGT[g]; cap=W.CAP[g]
            iso=W.fit_powercurve(FR[g],tgt,cap)
            full_frames[g]=(W.with_pc(FR[g],iso),W.with_pc(FR_TE[g],iso))
        test_pred=blend_predict(full_frames)
        out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
        for g in GROUPS:
            out[f"kpx_group_{g}"]=apply_post(g,full_frames[g][1],test_pred[g])
        assert out.shape[0]==8760
        for g in GROUPS:
            c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all() and c.notna().all()
        out.to_csv("submission_v5.csv",index=False); print("saved submission_v5.csv")
    else:
        print("holdout에서 v4를 넘지 못함 → v4 유지")
else:
    print("조건화 변형 미채택 → v4(submission_v4.csv) 유지가 최종 결론")

조건화 변형 미채택 → v4(submission_v4.csv) 유지가 최종 결론
